# GPT-2 Medium Fine-Tuning for CNN/DailyMail Abstractive Summarization
**Course:** CSEg5306 — Introduction to NLP, Adama Science and Technology University

This notebook fine-tunes `gpt2-medium` (355M params) on the CNN/DailyMail
news-summarization corpus using the **TL;DR prefix** strategy and **article-masked
cross-entropy loss** (gradient flows only through summary tokens).

## Before you run
1. **Settings → Accelerator:** `GPU T4 x2`
2. **Settings → Internet:** `On` (needed first run only — fetches model from HF)
3. **Add Data:** attach `gowrishankarp/newspaper-text-summarization-cnn-dailymail`
4. Run cells top-to-bottom. Expected runtime: ~2.5–3 h for 30k × 2 epochs on 1×T4.

The 2nd T4 is left idle on purpose — wiring DDP across both adds complexity for
modest speedup. If you want to use both, switch to `accelerate launch` from a script.

In [ ]:
!pip -q install -U \
    "transformers>=4.46,<4.50" \
    "datasets>=3.0" \
    "accelerate>=1.0" \
    evaluate rouge_score nltk

In [ ]:
import os, re, glob, json, random, gc
import numpy as np, torch

SEED            = 42
MODEL_NAME      = "gpt2-medium"
MAX_LEN         = 1024
SUMMARY_BUDGET  = 160          # tokens reserved for summary; rest goes to article
TRAIN_SUBSET    = 30_000        # full train is 287,113 — start small, scale up
VAL_SUBSET      = 1_000
TEST_SUBSET     = 500           # ROUGE eval sample

OUT_DIR   = "/kaggle/working"
CKPT_DIR  = f"{OUT_DIR}/ckpt"
FINAL_DIR = f"{OUT_DIR}/gpt2-medium-cnndm"

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

assert torch.cuda.is_available(), "Turn on GPU in Settings → Accelerator"
print("GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

## 1. Locate the Kaggle dataset CSVs

In [ ]:
DATA_ROOT = "/kaggle/input/newspaper-text-summarization-cnn-dailymail"
csv_paths = sorted(glob.glob(f"{DATA_ROOT}/**/*.csv", recursive=True))
print(csv_paths)

def pick(keyword):
    hits = [p for p in csv_paths if keyword in os.path.basename(p).lower()]
    assert hits, f"no csv matching '{keyword}' under {DATA_ROOT}"
    return hits[0]

TRAIN_CSV = pick("train")
VAL_CSV   = pick("valid")
TEST_CSV  = pick("test")
print("train:", TRAIN_CSV)
print("val  :", VAL_CSV)
print("test :", TEST_CSV)

## 2. Load and inspect

In [ ]:
from datasets import load_dataset

raw = load_dataset("csv", data_files={
    "train":      TRAIN_CSV,
    "validation": VAL_CSV,
    "test":       TEST_CSV,
})
print(raw)
print("\n--- sample article (first 400 chars) ---")
print(raw["train"][0]["article"][:400])
print("\n--- reference summary ---")
print(raw["train"][0]["highlights"])

## 3. Cleaning

Strip recurring journalistic boilerplate (e.g. `(CNN) -- `, `By ... Updated ...`),
collapse whitespace. We **keep Unicode** — GPT-2 uses byte-level BPE so curly
quotes and em-dashes encode fine, and stripping them destroys cues.

In [ ]:
LEAD_RE = re.compile(r"^\s*\(?[A-Z][A-Za-z\.\s]{0,30}\)?\s*--\s*")
BYLINE_RE = re.compile(r"By [A-Z][A-Za-z\.\s]{1,80}? (?:Reporter|Updated|PUBLISHED).*?\n", re.DOTALL)
WS_RE = re.compile(r"\s+")

def clean(text):
    if not text: return ""
    text = BYLINE_RE.sub("", text)
    text = LEAD_RE.sub("", text)
    text = WS_RE.sub(" ", text).strip()
    return text

def preprocess(ex):
    return {"article": clean(ex["article"]), "highlights": clean(ex["highlights"])}

raw = raw.map(preprocess, num_proc=2)

raw["train"]      = raw["train"].shuffle(seed=SEED).select(range(min(TRAIN_SUBSET, len(raw["train"]))))
raw["validation"] = raw["validation"].shuffle(seed=SEED).select(range(min(VAL_SUBSET, len(raw["validation"]))))
raw["test"]       = raw["test"].shuffle(seed=SEED).select(range(min(TEST_SUBSET, len(raw["test"]))))
print({k: len(v) for k, v in raw.items()})

## 4. Tokenizer + TL;DR prefix strategy

Each example becomes one continuous sequence:

```
<article tokens>  \n\nTL;DR:\n  <summary tokens>  <|endoftext|>
```

Critical: **labels for the article and separator are set to `-100`** so PyTorch
cross-entropy ignores them. Gradient only flows through summary tokens — the
model learns to *summarize*, not to memorize news prose.

Note: GPT-2 has no pad token by default, so we point pad → eos (standard practice).

In [ ]:
from transformers import GPT2TokenizerFast

tok = GPT2TokenizerFast.from_pretrained(MODEL_NAME)
tok.pad_token = tok.eos_token

SEP = "\n\nTL;DR:\n"
sep_ids = tok.encode(SEP, add_special_tokens=False)
print(f"separator: {sep_ids}  ({len(sep_ids)} tokens)")

In [ ]:
def encode(ex):
    article_budget = MAX_LEN - SUMMARY_BUDGET - len(sep_ids) - 1   # -1 for trailing EOS
    art_ids = tok.encode(ex["article"],    add_special_tokens=False)[:article_budget]
    sum_ids = tok.encode(ex["highlights"], add_special_tokens=False)[:SUMMARY_BUDGET]

    input_ids = art_ids + sep_ids + sum_ids + [tok.eos_token_id]
    n_prompt  = len(art_ids) + len(sep_ids)
    labels    = [-100] * n_prompt + sum_ids + [tok.eos_token_id]
    return {"input_ids": input_ids, "labels": labels}

drop_cols = raw["train"].column_names
ds = raw.map(encode, remove_columns=drop_cols, num_proc=2)
print(ds)
print("first sample length:", len(ds["train"][0]["input_ids"]))
print("first 30 label ids :", ds["train"][0]["labels"][:30])
print("last 30 label ids  :", ds["train"][0]["labels"][-30:])

## 5. Custom data collator

Pads `input_ids` with eos, `attention_mask` with 0, `labels` with -100. The
default HF collator doesn't handle our prompt-masked labels correctly.

In [ ]:
class PromptMaskedCollator:
    def __init__(self, pad_id):
        self.pad_id = pad_id

    def __call__(self, batch):
        max_len = max(len(b["input_ids"]) for b in batch)
        ids, mask, lbl = [], [], []
        for b in batch:
            n = len(b["input_ids"]); pad = max_len - n
            ids.append(b["input_ids"] + [self.pad_id] * pad)
            mask.append([1]*n + [0]*pad)
            lbl.append(b["labels"]    + [-100]        * pad)
        return {
            "input_ids":      torch.tensor(ids,  dtype=torch.long),
            "attention_mask": torch.tensor(mask, dtype=torch.long),
            "labels":         torch.tensor(lbl,  dtype=torch.long),
        }

collator = PromptMaskedCollator(pad_id=tok.eos_token_id)

## 6. Load `gpt2-medium`

- 24 layers, 1024 hidden, 16 heads, 355M params.
- `use_cache=False` is **required** with gradient checkpointing — they conflict.
- Gradient checkpointing trades ~30% extra compute for ~3× lower activation memory; without it we OOM on T4.

In [ ]:
from transformers import GPT2LMHeadModel

model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)
model.config.use_cache = False
model.gradient_checkpointing_enable()

n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"model: {MODEL_NAME}  |  {n_params:.1f}M params")

## 7. Training

Why these hyperparameters:
- `lr=3e-5`: gpt2-medium destabilizes above 5e-5 on summarization.
- `cosine` + 3% warmup: standard for fine-tuning a pretrained LM.
- `fp16=True` (not bf16): T4 is Turing arch (compute 7.5) — has FP16 tensor cores
  but NO native BF16. Forcing BF16 falls back to slow software emulation.
- `gradient_accumulation_steps=16`: effective batch = 16. Memory caps real batch at 1.
- Eval set capped at 200 to keep eval fast (we score ROUGE properly later).

In [ ]:
from transformers import TrainingArguments, Trainer

args = TrainingArguments(
    output_dir=CKPT_DIR,
    overwrite_output_dir=True,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=16,

    num_train_epochs=2,
    learning_rate=3e-5,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    weight_decay=0.1,
    max_grad_norm=1.0,

    fp16=True,
    optim="adamw_torch",
    gradient_checkpointing=True,

    logging_steps=25,
    save_strategy="steps", save_steps=1000, save_total_limit=1,
    eval_strategy="steps", eval_steps=1000,

    dataloader_num_workers=2,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=ds["train"],
    eval_dataset=ds["validation"].select(range(min(200, len(ds["validation"])))),
    data_collator=collator,
)

In [ ]:
trainer.train()

In [ ]:
trainer.save_model(FINAL_DIR)
tok.save_pretrained(FINAL_DIR)
print("saved →", FINAL_DIR)
gc.collect(); torch.cuda.empty_cache()

## 8. Generation

In [ ]:
@torch.no_grad()
def summarize(article, max_new_tokens=128):
    article_budget = MAX_LEN - max_new_tokens - len(sep_ids) - 4
    art_ids = tok.encode(clean(article), add_special_tokens=False)[:article_budget]
    prompt_ids = torch.tensor([art_ids + sep_ids], device=model.device)

    out = model.generate(
        prompt_ids,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        top_p=0.92,
        temperature=0.8,
        repetition_penalty=1.2,
        no_repeat_ngram_size=3,
        eos_token_id=tok.eos_token_id,
        pad_token_id=tok.eos_token_id,
    )
    gen = out[0, prompt_ids.shape[1]:]
    return tok.decode(gen, skip_special_tokens=True).strip()

model.config.use_cache = True   # turn KV cache back on for inference speed
model.eval()

### Quick qualitative check

In [ ]:
for i in range(3):
    ex = raw["test"][i]
    print(f"\n===== Example {i} =====")
    print("ARTICLE :", ex["article"][:400], "...")
    print("\nREFERENCE:", ex["highlights"])
    print("\nGENERATED:", summarize(ex["article"]))

## 9. ROUGE evaluation

`rouge-score` uses Porter stemming when `use_stemmer=True`, which matches the
official Hermann et al. CNN/DM convention.

500 examples × ~3s each ≈ ~25 min on T4. Drop `TEST_SUBSET` further if you need
to budget GPU time.

In [ ]:
import evaluate
rouge = evaluate.load("rouge")

preds, refs = [], []
N = min(TEST_SUBSET, len(raw["test"]))
for i in range(N):
    ex = raw["test"][i]
    preds.append(summarize(ex["article"]))
    refs.append(ex["highlights"])
    if (i + 1) % 25 == 0:
        print(f"{i+1}/{N}")

scores = rouge.compute(predictions=preds, references=refs, use_stemmer=True)
scores = {k: round(float(v), 4) for k, v in scores.items()}
print(json.dumps(scores, indent=2))

with open(f"{OUT_DIR}/rouge.json", "w") as f:
    json.dump(scores, f, indent=2)
with open(f"{OUT_DIR}/predictions.jsonl", "w") as f:
    for p, r in zip(preds, refs):
        f.write(json.dumps({"prediction": p, "reference": r}) + "\n")
print("wrote rouge.json and predictions.jsonl")

## 10. Variant runs (for the comparative analysis)

### Scratch baseline (Protocol A)
Replace the **model load cell** with:
```python
from transformers import GPT2Config, GPT2LMHeadModel
cfg = GPT2Config(vocab_size=50257, n_positions=1024, n_ctx=1024,
                 n_embd=512, n_layer=6, n_head=8)
model = GPT2LMHeadModel(cfg)
model.config.use_cache = False
model.gradient_checkpointing_enable()
```
And bump LR in `TrainingArguments`:
```python
learning_rate=3e-4, warmup_ratio=0.05,
```

### Upgrade to gpt2-large + LoRA
Add `peft` to the install cell, then:
```python
from peft import LoraConfig, get_peft_model
model = GPT2LMHeadModel.from_pretrained("gpt2-large")
model.config.use_cache = False
model.gradient_checkpointing_enable()
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, target_modules=["c_attn"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
))
model.print_trainable_parameters()
```

### Zero-shot baseline (no fine-tuning)
Skip the `trainer.train()` cell entirely and run §8–9. This gives the third bar
in your comparison and quantifies the actual gain from fine-tuning vs raw GPT-2.

## Realistic ROUGE targets
| Model | R-1 | R-2 | R-L |
|---|---|---|---|
| gpt2-small zero-shot      | ~24 | ~7  | ~17 |
| gpt2-medium zero-shot     | ~26 | ~8  | ~18 |
| gpt2-medium fine-tuned    | **~36** | **~15** | **~26** |
| gpt2-large + LoRA         | ~38 | ~16 | ~28 |
| Scratch (6L, 30k samples) | ~12 | ~2  | ~10 |

Update §9.2 of your PDF with these — the >30 R-L claim for the 124M model is too optimistic.